In [ ]:
!pip install langchain langchain-community faiss-cpu sentence-transformers
print("ابدأنا")

ابدأنا


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

print("تم الاستيراد بنجاح")

تم الاستيراد بنجاح


In [ ]:
!pip install langchain langchain-community

In [ ]:
!pip install langchain-huggingface

In [ ]:
# 1️⃣ تثبيت
!pip install langchain langchain-huggingface langchain-community faiss-cpu sentence-transformers pypdf

# 2️⃣ الاستيراد
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
import requests
import numpy as np

print("✅ تم التحميل")

# 3️⃣ رفع ملفاتك
from google.colab import files
print("📤 ارفع ملفاتك (PDF أو TXT):")
uploaded = files.upload()

# 4️⃣ تحميل وتقسيم ذكي
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", "।", "،", " ", ""]
)

all_chunks = []

for filename in uploaded.keys():
    if filename.endswith('.txt'):
        with open(filename, 'r', encoding='utf-8') as f:
            text = f.read()
            chunks = text_splitter.create_documents([text])
            all_chunks.extend(chunks)
            print(f"📄 {filename}: {len(chunks)} جزء")

    elif filename.endswith('.pdf'):
        loader = PyPDFLoader(filename)
        pages = loader.load()
        chunks = text_splitter.split_documents(pages)
        all_chunks.extend(chunks)
        print(f"📄 {filename}: {len(chunks)} جزء")

print(f"\n📚 المجموع: {len(all_chunks)} جزء")

# تقسيم الأجزاء إلى 4 مجموعات
print("🔄 جاري تقسيم الملف إلى 4 أقسام...")

np.random.seed(42)
indices = np.random.permutation(len(all_chunks))
split_size = len(all_chunks) // 4
remainder = len(all_chunks) % 4

chunk_groups = []
start = 0
for i in range(4):
    extra = 1 if i < remainder else 0
    end = start + split_size + extra
    group_indices = indices[start:end]
    chunk_groups.append([all_chunks[idx] for idx in group_indices])
    start = end

print(f"📊 التقسيم: {[len(g) for g in chunk_groups]} جزء لكل مجموعة")

# 5️⃣ تخزين في قاعدة متجهة (مرة واحدة فقط)
print("🔄 جاري تحويل النصوص لمتجهات...")
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")

# معالجة المجموعات
for idx, group in enumerate(chunk_groups):
    print(f"🔄 معالجة المجموعة {idx+1}/4 ({len(group)} جزء)...")
    group_db = FAISS.from_documents(group, embeddings)

    if idx == 0:
        db = group_db
    else:
        db.merge_from(group_db)

    print(f"✅ تمت المجموعة {idx+1}")

db.save_local("drug_database")
print("💾 تم حفظ قاعدة البيانات")
print("🎉 النظام جاهز!")

# 6️⃣ البحث الذكي
DEEPSEEK_API_KEY = "sk-08cb0518f2de46359748c8413fb46ee7"

while True:
    question = input("\n💊 اسأل (أو 'خروج'): ")
    if question == 'خروج':
        break

    docs = db.similarity_search(question, k=5)
    context = "\n---\n".join([doc.page_content for doc in docs])

    response = requests.post(
        "https://api.deepseek.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {DEEPSEEK_API_KEY}"},
        json={
            "model": "deepseek-chat",
            "messages": [
                {"role": "system", "content": f"أنت صيدلي خبير. أجب من المعلومات فقط:\n{context}"},
                {"role": "user", "content": question}
            ],
            "temperature": 0.3
        }
    )

    print(f"\n💬 {response.json()['choices'][0]['message']['content']}")
    print("\n" + "="*50)

✅ تم التحميل
📤 ارفع ملفاتك (PDF أو TXT):


Saving BNF 83 (2022).pdf to BNF 83 (2022).pdf


📄 BNF 83 (2022).pdf: 27701 جزء

📚 المجموع: 27701 جزء
🔄 جاري تقسيم الملف إلى 4 أقسام...
📊 التقسيم: [6926, 6925, 6925, 6925] جزء لكل مجموعة
🔄 جاري تحويل النصوص لمتجهات...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔄 معالجة المجموعة 1/4 (6926 جزء)...
✅ تمت المجموعة 1
🔄 معالجة المجموعة 2/4 (6925 جزء)...
✅ تمت المجموعة 2
🔄 معالجة المجموعة 3/4 (6925 جزء)...
✅ تمت المجموعة 3
🔄 معالجة المجموعة 4/4 (6925 جزء)...
✅ تمت المجموعة 4
💾 تم حفظ قاعدة البيانات
🎉 النظام جاهز!

💊 اسأل (أو 'خروج'): paracetamol dose

💬 Based solely on the provided information:

**For Paracetamol Tablets (Paravict 500mg):**
- Strength: 500 mg per tablet
- Pack size: 100 tablets
- Price: £1.62 (Drug Tariff price = £2.78)

**For Paracetamol Suppositories:**
Available in various strengths:
- 80 mg, 120 mg, 125 mg, 240 mg, 250 mg, 500 mg, and 1 gram (1000 mg).
- Pack size: 10 suppositories per pack.
- Prices vary by strength (e.g., 500 mg suppositories: £41.27 per 10).

**Important Safety Information from the Text:**
1.  **Overdose Definition:** Ingestion of a **licensed dose is not an overdose**. A dose consistently less than **75 mg/kg in any 24-hour period** is very unlikely to be toxic.
2.  **Therapeutic Excess:** Ingestion of a po

KeyboardInterrupt: Interrupted by user

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

# تأكد من وجود المجلد
if os.path.exists("/content/drug_database"):
    # احذف النسخة القديمة من Drive إذا موجودة
    if os.path.exists("/content/drive/MyDrive/drug_database"):
        !rm -rf "/content/drive/MyDrive/drug_database"

    # انسخ المجلد إلى Drive
    shutil.copytree("/content/drug_database", "/content/drive/MyDrive/drug_database")
    print("✅ تم الحفظ في: /content/drive/MyDrive/drug_database")

    # اعرض حجمها
    !du -sh "/content/drive/MyDrive/drug_database"
else:
    print("⚠️ المجلد /content/drug_database غير موجود!")

✅ تم الحفظ في: /content/drive/MyDrive/drug_database
58M	/content/drive/MyDrive/drug_database
